# COMPX525 A2 B1 Synthetic BirdNET + ML

Train ML classifier on real train BirdNET features plus accepted synthetic BirdNET features.


In [ ]:
from __future__ import annotations

import ast
import json
import time
from pathlib import Path
from typing import Sequence

import numpy as np
import pandas as pd
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import roc_auc_score
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 525
CLASSIFIER_MAX_ITER = 20
TRAIN_AUDIO_ONLY_SYNTHETIC = True
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
INPUT_ROOT = Path("/kaggle/input")

def find_data_root() -> Path:
    required = ["train.csv", "taxonomy.csv", "sample_submission.csv"]
    candidates = [Path("/kaggle/input/competitions/birdclef-2026"), Path("/kaggle/input/birdclef-2026"), INPUT_ROOT]
    for root in candidates:
        if all((root / x).exists() for x in required):
            return root
        if root.exists():
            for child in sorted(root.glob("*")):
                if child.is_dir() and all((child / x).exists() for x in required):
                    return child
    raise FileNotFoundError("BirdCLEF competition data not found")

def parse_secondary_labels(value) -> list[str]:
    if pd.isna(value):
        return []
    value = str(value).strip()
    if not value:
        return []
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [str(x) for x in parsed]
    except Exception:
        return []
    return []

def encode(labels: Sequence[str], label_to_idx: dict[str, int]) -> np.ndarray:
    y = np.zeros(len(label_to_idx), dtype=np.float32)
    for label in labels:
        if label in label_to_idx:
            y[label_to_idx[label]] = 1.0
    return y

def kaggle_macro_auc_skip_empty(y_true: np.ndarray, y_score: np.ndarray) -> float:
    aucs = []
    for i in range(y_true.shape[1]):
        if y_true[:, i].sum() == 0:
            continue
        if y_true[:, i].sum() == len(y_true):
            continue
        aucs.append(roc_auc_score(y_true[:, i], y_score[:, i]))
    return float(np.mean(aucs)) if aucs else float("nan")

def discover_real_feature_files() -> list[Path]:
    files = sorted(INPUT_ROOT.rglob("birdnet_all_chunk*_clip_features.npz"))
    print("real feature files", len(files))
    for p in files[:20]:
        print("real feature", p)
    return files

def discover_synthetic_feature_file() -> Path:
    files = sorted(INPUT_ROOT.rglob("synthetic_birdnet_clip_features.npz"))
    if len(files) != 1:
        raise ValueError(f"Expected one synthetic feature file, found {len(files)}: {files}")
    print("synthetic feature", files[0])
    return files[0]

def load_feature_files(files: Sequence[Path]) -> tuple[np.ndarray, pd.DataFrame]:
    feature_blocks = []
    meta_blocks = []
    for path in files:
        with np.load(path, allow_pickle=True) as data:
            features = data["features"].astype(np.float32)
            filenames = data["filenames"].astype(str)
            labels = data["labels"].astype(str) if "labels" in data else np.array([str(x).split("/", 1)[0] for x in filenames])
        feature_blocks.append(features)
        meta_blocks.append(pd.DataFrame({"filename": filenames, "feature_primary_label": labels, "feature_file": str(path)}))
    features = np.concatenate(feature_blocks, axis=0)
    metadata = pd.concat(meta_blocks, ignore_index=True)
    keep = ~metadata["filename"].duplicated(keep="first").to_numpy()
    print("feature rows", len(metadata), "duplicates dropped", int((~keep).sum()))
    return features[keep], metadata.loc[keep].reset_index(drop=True)

DATA_ROOT = find_data_root()
train_df = pd.read_csv(DATA_ROOT / "train.csv")
taxonomy_df = pd.read_csv(DATA_ROOT / "taxonomy.csv")
sample_submission_df = pd.read_csv(DATA_ROOT / "sample_submission.csv")
train_audio_label_set = set(train_df["primary_label"].astype(str))
label_cols = [c for c in sample_submission_df.columns if c != "row_id"]
taxonomy_labels = taxonomy_df["primary_label"].astype(str).tolist()
assert label_cols == taxonomy_labels
label_to_idx = {label: i for i, label in enumerate(label_cols)}

split_manifest_files = sorted(INPUT_ROOT.rglob("train_audio_split_manifest.csv"))
if not split_manifest_files:
    raise FileNotFoundError("train_audio_split_manifest.csv not found in attached chunk outputs")
split_df = pd.read_csv(split_manifest_files[0])
split_df["primary_label"] = split_df["primary_label"].astype(str)
print("split manifest", split_manifest_files[0], split_df.shape)

real_features, real_meta = load_feature_files(discover_real_feature_files())
join_cols = ["filename", "primary_label", "secondary_labels", "split"]
real_joined = real_meta.merge(split_df[join_cols], how="left", on="filename")
missing_split = int(real_joined["split"].isna().sum())
if missing_split:
    print("dropping real rows missing split", missing_split)
keep = real_joined["split"].notna().to_numpy()
real_joined = real_joined.loc[keep].reset_index(drop=True)
real_features = real_features[keep]

real_labels = []
for _, row in real_joined.iterrows():
    labels = [str(row["primary_label"])]
    labels.extend(parse_secondary_labels(row.get("secondary_labels")))
    real_labels.append(sorted(set(x for x in labels if x in label_to_idx)))
real_y = np.stack([encode(labels, label_to_idx) for labels in real_labels]).astype(np.float32)

synthetic_feature_path = discover_synthetic_feature_file()
with np.load(synthetic_feature_path, allow_pickle=True) as data:
    synthetic_features = data["features"].astype(np.float32)
    synthetic_filenames = data["filenames"].astype(str)
    synthetic_labels = data["labels"].astype(str)
    synthetic_ids = data["synthetic_ids"].astype(str) if "synthetic_ids" in data else np.array([Path(x).stem for x in synthetic_filenames])
if TRAIN_AUDIO_ONLY_SYNTHETIC:
    synthetic_keep = np.array([label in train_audio_label_set for label in synthetic_labels], dtype=bool)
    print("train_audio_only synthetic filter kept", int(synthetic_keep.sum()), "of", int(len(synthetic_keep)))
    synthetic_features = synthetic_features[synthetic_keep]
    synthetic_filenames = synthetic_filenames[synthetic_keep]
    synthetic_labels = synthetic_labels[synthetic_keep]
    synthetic_ids = synthetic_ids[synthetic_keep]
synthetic_y = np.stack([encode([label], label_to_idx) for label in synthetic_labels]).astype(np.float32)
synthetic_meta = pd.DataFrame({
    "filename": synthetic_filenames,
    "synthetic_id": synthetic_ids,
    "primary_label": synthetic_labels,
    "split": "train",
    "feature_file": str(synthetic_feature_path),
})

train_mask = real_joined["split"].eq("train").to_numpy()
val_mask = real_joined["split"].eq("val").to_numpy()
x_train = np.concatenate([real_features[train_mask], synthetic_features], axis=0)
y_train = np.concatenate([real_y[train_mask], synthetic_y], axis=0)
x_val = real_features[val_mask]
y_val = real_y[val_mask]

print("x_train", x_train.shape, "x_val", x_val.shape)
print("synthetic_features", synthetic_features.shape)
print("train positive classes", int((y_train.sum(axis=0) > 0).sum()), "val positive classes", int((y_val.sum(axis=0) > 0).sum()))

classifier = make_pipeline(
    StandardScaler(),
    OneVsRestClassifier(
        SGDClassifier(
            loss="log_loss",
            alpha=1e-4,
            max_iter=CLASSIFIER_MAX_ITER,
            tol=1e-3,
            class_weight="balanced",
            random_state=SEED,
        ),
        n_jobs=-1,
    ),
)
start = time.time()
classifier.fit(x_train, y_train)
fit_elapsed = time.time() - start
val_pred = classifier.predict_proba(x_val).astype(np.float32)
val_auc = kaggle_macro_auc_skip_empty(y_val, val_pred)

val_pred_df = pd.DataFrame(val_pred, columns=label_cols)
val_pred_df.insert(0, "filename", real_joined.loc[val_mask, "filename"].to_numpy())
val_pred_df.to_csv(WORK_DIR / "synthetic_birdnet_ml_val_predictions.csv", index=False)

real_train_labels = set(np.where(real_y[train_mask].sum(axis=0) > 0)[0].tolist())
synthetic_train_labels = set(np.where(synthetic_y.sum(axis=0) > 0)[0].tolist())
added_positive_classes = sorted(synthetic_train_labels - real_train_labels)

summary = {
    "experiment": "B1 synthetic-enriched BirdNET embeddings + ML classifier",
    "classifier": "StandardScaler + OneVsRestClassifier(SGDClassifier log_loss)",
    "classifier_max_iter": int(CLASSIFIER_MAX_ITER),
    "train_audio_only_synthetic": bool(TRAIN_AUDIO_ONLY_SYNTHETIC),
    "real_feature_rows": int(len(real_joined)),
    "real_train_rows": int(train_mask.sum()),
    "real_val_rows": int(val_mask.sum()),
    "synthetic_train_rows": int(len(synthetic_features)),
    "train_rows_total": int(len(x_train)),
    "val_rows": int(len(x_val)),
    "feature_dim": int(x_train.shape[1]),
    "labels": int(len(label_cols)),
    "fit_elapsed_seconds": float(fit_elapsed),
    "val_macro_auc_skip_empty": float(val_auc),
    "train_positive_classes": int((y_train.sum(axis=0) > 0).sum()),
    "val_positive_classes": int((y_val.sum(axis=0) > 0).sum()),
    "synthetic_positive_classes": int((synthetic_y.sum(axis=0) > 0).sum()),
    "added_positive_classes_from_synthetic": [label_cols[i] for i in added_positive_classes],
    "added_positive_classes_from_synthetic_count": int(len(added_positive_classes)),
}
(WORK_DIR / "synthetic_birdnet_ml_summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")
real_joined.to_csv(WORK_DIR / "synthetic_birdnet_ml_real_feature_metadata.csv", index=False)
synthetic_meta.to_csv(WORK_DIR / "synthetic_birdnet_ml_synthetic_feature_metadata.csv", index=False)
print(json.dumps(summary, indent=2))
